# Chess · Deploy
Choose **Runtime → Run all** and allow Drive access. This notebook loads the saved champion, starts the API, opens a Cloudflare tunnel, tests it, and **finishes with your URL**. It never trains. If no champion exists yet, run **[train.ipynb](train.ipynb)** first.

A CPU runtime works for deployment; GPU is optional. Leave the Colab runtime connected while playing. This notebook does not prevent Colab idle/runtime limits. Code is collapsed but still executes normally.

Supports the original dense champion, evolved classical parameters and exported sparse neural/hybrid models. The saved bundle includes its engine code; after updating train.ipynb, run training/export again and restart this notebook to load that updated bundle. Existing published snapshots keep their original code.

The default training workflow now exports a **full_pipeline** champion. Its saved evaluator, evolved classical weights, neural weights and validated search options are loaded together. The deployment notebook never starts a training or evaluation campaign.

Dense exports carry their own architecture, including the 30,017,409-parameter network; no deployment size setting is needed.

Loads only the new **MyDrive/chess_30m/models/** champion. No older model folder is used as a fallback. WEBSITE_ORIGINS defaults to * so the API accepts requests from any website. Public connection tests retry every 3 seconds until successful; interrupt the cell to stop and close the tunnel.

**Memory/device:** `API_DEVICE = auto` loads one dense network on GPU when available and falls back to CPU if allocation fails. Choose `cpu` to use CPU even in a GPU runtime. Deployment ignores ES population memory/headroom settings. This also fixes previously exported champions without retraining or modifying their saved files. A separate deployment runtime avoids competing with training for GPU memory.


In [3]:
#@title 1 · Load the saved champion from Drive { display-mode: "form" }
# BEGIN GENERATED INFERENCE SUPPORT
INFERENCE_SOURCE = r'''"""Single-model inference for dense exports, including older notebook bundles.

Training population budgets describe an entire ES population and allocator-wide
memory; neither is a suitable admission check for one inference model. Load only
its layer tensors and let allocation failure drive an explicit CPU fallback.
"""


def _place(bank, device):
    """One packed layer allocation; no perturbations, optimizer or GPU mean copy."""
    torch = bank.torch
    bank.device = torch.device(device)
    bank.dtype = torch.float16 if bank.device.type == 'cuda' else torch.float32
    bank.layers = []
    bank.prior = None
    with torch.inference_mode():
        packed = bank.mean.to(device=bank.device, dtype=bank.dtype).reshape(1, -1)
        for offset, inputs, outputs in bank.layout:
            bank.layers.append((packed[:, offset:offset + inputs*outputs].reshape(1, inputs, outputs),
                                packed[:, offset + inputs*outputs:offset + inputs*outputs + outputs].reshape(1, 1, outputs)))
        bank.prior = torch.as_tensor(bank._inference_prior, device=bank.device, dtype=bank.dtype)


def _release_cuda_layers(bank):
    bank.layers = []
    bank.prior = None
    # Call outside the OOM except block so its traceback no longer holds tensors.
    bank.torch.cuda.empty_cache()


def initialize(bank, cfg, state, layout_function, device=None):
    import torch
    bank.torch, bank.cfg = torch, dict(cfg)
    requested = device or cfg.get('gpu_inference_device', 'auto')
    if requested not in ('auto', 'cpu', 'cuda'):
        raise ValueError('Inference device must be auto, cpu, or cuda.')
    bank.inference_device_policy = requested
    bank.layout, bank.count = layout_function(cfg)
    # Keep the original CPU tensor for fallback; never modify saved weights.
    bank.mean = state['mean'].detach().cpu()
    if bank.mean.ndim != 1 or bank.mean.numel() != bank.count:
        raise ValueError('Dense model weights do not match the exported architecture.')
    bank._inference_prior = state['prior']
    bank.scale, bank.budget = None, None
    bank.seeds, bank.noise, bank.hall_ids = [], [], []
    bank.mean_id = bank.champion_id = bank.default_id = 0
    bank.pairs, bank.sigma = 0, 0.
    bank.batch_rows = cfg['gpu_batch_rows']
    bank.positions, bank.forward_seconds = 0, 0.
    target = 'cpu'
    if requested != 'cpu':
        if torch.cuda.is_available():
            target = 'cuda'
        elif requested == 'cuda':
            raise RuntimeError('CUDA is unavailable. Select API_DEVICE="cpu" or "auto", or use a GPU runtime.')
    fallback = False
    try:
        _place(bank, target)
    except torch.cuda.OutOfMemoryError:
        if target != 'cuda' or requested != 'auto':
            raise
        fallback = True
    if fallback:
        _release_cuda_layers(bank)
        _place(bank, 'cpu')
        print('GPU memory is insufficient for inference; using CPU with the same saved weights.', flush=True)
    print(f'Inference device: {bank.device} | 1 network | {bank.count:,} parameters', flush=True)


def population_class(original, layout_function, device=None):
    """Replace only inference initialization; preserve the export's math/search.

    Deployment can install this adapter in a verified old module in memory. No
    artifact rewrite, checksum bypass, checkpoint migration or retraining needed.
    """
    original = getattr(original, '_chess_original_population', original)
    default_device = device

    class InferenceAwarePopulation(original):
        _chess_original_population = original

        def __init__(self, cfg, state, device=None):
            if cfg.get('gpu_inference_only'):
                initialize(self, cfg, state, layout_function, device=device or default_device)
            else:
                super().__init__(cfg, state, device=device)

        def evaluate(self, requests):
            try:
                return super().evaluate(requests)
            except self.torch.cuda.OutOfMemoryError:
                if (not self.cfg.get('gpu_inference_only') or self.device.type != 'cuda'
                        or self.inference_device_policy != 'auto'):
                    raise
            # The original evaluator already tried smaller batches. An HTTP
            # deadline/error is not an OOM and must propagate without fallback.
            _release_cuda_layers(self)
            _place(self, 'cpu')
            print('GPU inference ran out of memory; continuing on CPU with the same saved weights.', flush=True)
            return super().evaluate(requests)

    return InferenceAwarePopulation
'''
# END GENERATED INFERENCE SUPPORT
# Standalone deployment: run this section after exporting, without rerunning training.
API_DEVICE = 'auto' #@param ['auto', 'cpu', 'cuda']
# Auto tries one model on CUDA and falls back to CPU on allocation failure.
if API_DEVICE not in ('auto', 'cpu', 'cuda'):
    raise ValueError('API_DEVICE must be auto, cpu, or cuda.')
import sys, subprocess, importlib.util, json, time, threading
from pathlib import Path
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'fastapi', 'uvicorn', 'nest_asyncio', 'python-chess', 'numpy', 'requests'])
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
MODEL_ROOT = Path('/content/drive/MyDrive/chess_30m/models')
manifest_path = MODEL_ROOT / 'latest_champion.json'
if not manifest_path.is_file():
    raise FileNotFoundError('No fresh champion exists. Run the updated train.ipynb first (MyDrive/chess_30m).')
import hashlib
manifest = json.loads(manifest_path.read_text())
API_MODELS = (MODEL_ROOT / manifest['folder']).resolve()
if not API_MODELS.is_relative_to(MODEL_ROOT.resolve()) or API_MODELS == MODEL_ROOT.resolve():
    raise ValueError('Champion folder must be inside the models directory.')
required_files = {'cristopher_engine.py', 'cristopher_genome.json'}
if not required_files <= set(manifest.get('sha256', {})):
    raise RuntimeError('Champion manifest is incomplete.')
for name, checksum in manifest['sha256'].items():
    path = API_MODELS / name
    if path.is_symlink() or path.resolve().parent != API_MODELS or not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != checksum:
        raise RuntimeError('Incomplete or damaged champion snapshot. Run train.ipynb to publish again.')
for name in ('cristopher_engine.py', 'cristopher_genome.json'):
    if not (API_MODELS / name).is_file():
        raise FileNotFoundError(f'Missing {name}. Run train.ipynb first to save a champion to Drive.')
api_metadata = json.loads((API_MODELS / 'cristopher_genome.json').read_text())
if api_metadata.get('weights'):
    if api_metadata['weights'] not in manifest['sha256'] or not (API_MODELS / api_metadata['weights']).is_file():
        raise FileNotFoundError('Missing exported neural weights. Run train.ipynb first to save the champion.')
    if Path(api_metadata['weights']).suffix == '.pt' and importlib.util.find_spec('torch') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch'])
# Re-running deployment first closes its previous server/tunnel, then loads the latest bundle.
if callable(globals().get('stop_tunnel')):
    stop_tunnel()
if callable(globals().get('stop_server')):
    stop_server()
# Free the previous deployment model before loading another copy on a rerun.
# Training objects/checkpoints, if present in this runtime, are not touched.
globals().pop('API_ENGINE', None)
globals().pop('API_MODULE', None)
sys.modules.pop('cristopher_api_engine', None)
import gc
gc.collect()
if 'torch' in sys.modules and sys.modules['torch'].cuda.is_available():
    sys.modules['torch'].cuda.empty_cache()
spec = importlib.util.spec_from_file_location('cristopher_api_engine', API_MODELS / 'cristopher_engine.py')
API_MODULE = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = API_MODULE
spec.loader.exec_module(API_MODULE)
# Apply inference support to verified older exports in memory, without rewriting Drive files.
import types
API_INFERENCE = types.ModuleType('chess_deployment_inference')
exec(compile(INFERENCE_SOURCE, '<deployment-inference>', 'exec'), API_INFERENCE.__dict__)
if hasattr(API_MODULE, 'GpuPopulation') and hasattr(API_MODULE, 'gpu_layout'):
    API_MODULE.GpuPopulation = API_INFERENCE.population_class(
        API_MODULE.GpuPopulation, API_MODULE.gpu_layout, device=API_DEVICE)
API_ENGINE = API_MODULE.CristopherEngine(API_MODELS / 'cristopher_genome.json')
API_VERSION = str(api_metadata.get('version', 'unknown'))
API_ENGINE_LOCK = threading.Lock()
if api_metadata.get('champion_generation') == -1:
    print('Notice: this is the initial saved baseline; training has not promoted a challenger yet.')
print(f'Loaded Chess ({API_VERSION}) from {API_MODELS}')
print('Search:', api_metadata.get('search_version', 'legacy export'))
print('Independent champion validation:', 'passed' if api_metadata.get('champion_verified') else 'not yet verified')
# No multipart dependency is needed: /move accepts JSON.
TUNNEL = 'cloudflare' #@param ['cloudflare', 'ngrok']
# * accepts browser requests from any origin. Exact origins remain an optional restriction.
WEBSITE_ORIGINS = '*' #@param {type:'string'}
API_TIME_CAPS = {'easy': 2.0, 'medium': 4.0, 'hard': 6.0}
# Keep this admin secret only in runtime memory. Never put it in browser code/URLs.
import secrets
API_ADMIN_TOKEN = globals().get('API_ADMIN_TOKEN') or secrets.token_urlsafe(32)

if api_metadata.get('pipeline_spec'):
    print('Full pipeline evaluator:', api_metadata['pipeline_spec']['kind'])
    print('Search configuration:', api_metadata['pipeline_spec']['options'])


Inference device: cuda | 1 network | 30,017,409 parameters
Loaded Chess (chess-full-pipeline-1) from /content/drive/MyDrive/chess_30m/models/deployments/629404768c78058d-d3f3ef06
Search: shared-alpha-beta-v2
Independent champion validation: not yet verified
Full pipeline evaluator: dense
Search configuration: {'pvs': True, 'aspiration': True, 'see': True, 'countermove': True, 'null_move': False, 'lmr': False, 'futility': False, 'check_extensions': False, 'tt_capacity': 32768, 'qdepth': 8, 'max_ply': 96}


In [4]:
#@title 2 · Prepare the API { display-mode: "form" }
# BEGIN GENERATED API SECURITY
SECURITY_SOURCE = r'''"""Bounded ASGI input handling and resource admission for the public chess API.

No application authentication secret is shipped to browsers. Origin checks limit
browser callers; global budgets also apply to clients that forge/omit Origin.
"""
import asyncio
from collections import OrderedDict
import ipaddress
import math
import threading
import time
from urllib.parse import urlsplit

from starlette.responses import JSONResponse


def allowed_origins(value):
    if value.strip() == '*':
        return ('*',)
    origins = []
    for item in value.split(','):
        item = item.strip().rstrip('/')
        if not item:
            continue
        url = urlsplit(item)
        local = url.hostname in ('localhost', '127.0.0.1', '::1')
        if (url.scheme not in (('http', 'https') if local else ('https',)) or not url.hostname
                or url.username or url.password or url.query or url.fragment or url.path
                or '*' in item):
            raise ValueError('Use * alone for any origin, or exact HTTPS origins without paths (localhost HTTP is allowed).')
        origins.append(item)
    if not origins:
        raise ValueError('At least one website origin is required.')
    return tuple(dict.fromkeys(origins))


class PublicPolicy:
    def __init__(self, origins, *, trust_cloudflare=False, body_bytes=16384, body_seconds=5.,
                 requests_per_minute=120, request_burst=30, moves_per_minute=20, move_burst=4,
                 global_requests_per_minute=600, global_request_burst=60,
                 global_moves_per_minute=60, global_move_burst=8, max_clients=4096,
                 max_searches=2, clock=time.monotonic):
        self.origins = allowed_origins(','.join(origins))
        self.trust_cloudflare = trust_cloudflare
        self.body_bytes, self.body_seconds = body_bytes, body_seconds
        self.max_clients, self.clock = max_clients, clock
        self.rates = ((requests_per_minute/60, request_burst), (moves_per_minute/60, move_burst))
        self.global_rates = ((global_requests_per_minute/60, global_request_burst),
                             (global_moves_per_minute/60, global_move_burst))
        self.clients = OrderedDict()
        self.global_buckets = {}
        self.lock = threading.Lock()
        self.search_slots = threading.BoundedSemaphore(max_searches)
        self.counts = {'rate_limited': 0, 'busy': 0, 'oversized': 0, 'blocked_origin': 0}

    def client_key(self, scope, headers):
        peer = (scope.get('client') or ('unknown', 0))[0]
        try:
            address = ipaddress.ip_address(peer)
        except ValueError:
            return peer
        # Uvicorn proxy-header rewriting is disabled, and listens only on loopback.
        # Trust this header only on the explicit Cloudflare tunnel configuration.
        if self.trust_cloudflare and address.is_loopback:
            try:
                address = ipaddress.ip_address(headers.get(b'cf-connecting-ip', b'').decode('ascii'))
            except (ValueError, UnicodeError):
                pass
        if address.version == 6:
            return str(ipaddress.ip_network(f'{address}/64', strict=False))
        return str(address)

    def admit(self, key, is_move):
        now = self.clock()
        with self.lock:
            while self.clients and next(iter(self.clients.values()))['seen'] < now-120:
                self.clients.popitem(last=False)
            if key not in self.clients:
                if len(self.clients) >= self.max_clients:
                    # Never evict live clients to reset their budget under IP churn.
                    self.counts['rate_limited'] += 1
                    return 60
                self.clients[key] = {'seen': now, 'buckets': {}}
            client = self.clients[key]
            client['seen'] = now
            self.clients.move_to_end(key)
            pending, retry = [], 0
            for buckets, limits in ((client['buckets'], self.rates), (self.global_buckets, self.global_rates)):
                for index in range(2 if is_move else 1):
                    rate, burst = limits[index]
                    tokens, last = buckets.get(index, (burst, now))
                    tokens = min(burst, tokens + max(0, now-last)*rate)
                    buckets[index] = (tokens, now)
                    if tokens < 1:
                        retry = max(retry, math.ceil((1-tokens)/rate))
                    pending.append((buckets, index, tokens))
            if retry:
                self.counts['rate_limited'] += 1
                return retry
            for buckets, index, tokens in pending:
                buckets[index] = (tokens-1, now)
            return 0

    def snapshot(self):
        with self.lock:
            return dict(self.counts)


class SecurityBoundary:
    """Read at most body_bytes BEFORE JSON parsing, including chunked requests."""
    def __init__(self, app, policy):
        self.app, self.policy = app, policy

    async def __call__(self, scope, receive, send):
        if scope['type'] != 'http':
            return await self.app(scope, receive, send)
        policy = self.policy
        headers = dict(scope['headers'])
        origin = headers.get(b'origin', b'').decode('latin1')
        async def secure_send(message):
            if message['type'] == 'http.response.start':
                message = dict(message)
                message['headers'] = list(message.get('headers', [])) + [
                    (b'x-content-type-options', b'nosniff'), (b'x-frame-options', b'DENY'),
                    (b'cache-control', b'no-store'), (b'referrer-policy', b'no-referrer'),
                    (b'content-security-policy', b"default-src 'none'; frame-ancestors 'none'")]
            await send(message)
        async def reject(status, text, extra=None):
            response_headers = dict(extra or {})
            if origin and '*' in policy.origins:
                response_headers['Access-Control-Allow-Origin'] = '*'
            elif origin in policy.origins:
                response_headers.update({'Access-Control-Allow-Origin': origin, 'Vary': 'Origin'})
            await JSONResponse({'error': text}, status_code=status, headers=response_headers)(scope, receive, secure_send)
        if origin and '*' not in policy.origins and origin not in policy.origins:
            with policy.lock:
                policy.counts['blocked_origin'] += 1
            return await reject(403, 'This website origin is not allowed.')
        key = policy.client_key(scope, headers)
        retry = policy.admit(key, scope['method'] == 'POST' and scope['path'] == '/move')
        if retry:
            return await reject(429, 'Too many requests. Please wait before retrying.', {'Retry-After': str(retry)})
        if scope['method'] == 'POST' and scope['path'] == '/move':
            if headers.get(b'content-encoding', b'identity').lower() != b'identity':
                return await reject(415, 'Compressed request bodies are not accepted.')
            if headers.get(b'content-type', b'').split(b';')[0].strip().lower() != b'application/json':
                return await reject(415, 'Use application/json.')
            lengths = [v for k, v in scope['headers'] if k.lower() == b'content-length']
            try:
                size = int(lengths[0]) if lengths else 0
                if len(lengths) > 1 or size < 0:
                    raise ValueError()
            except ValueError:
                return await reject(400, 'Invalid Content-Length.')
            if size > policy.body_bytes:
                with policy.lock:
                    policy.counts['oversized'] += 1
                return await reject(413, 'Request body is too large.')
            chunks, total = [], 0
            deadline = policy.clock() + policy.body_seconds
            while True:
                try:
                    message = await asyncio.wait_for(receive(), timeout=max(.001, deadline-policy.clock()))
                except asyncio.TimeoutError:
                    return await reject(408, 'Request body timed out.')
                if message['type'] == 'http.disconnect':
                    return
                chunk = message.get('body', b'')
                total += len(chunk)
                if total > policy.body_bytes:
                    with policy.lock:
                        policy.counts['oversized'] += 1
                    return await reject(413, 'Request body is too large.')
                chunks.append(chunk)
                if not message.get('more_body', False):
                    break
            body, consumed = b''.join(chunks), False
            async def replay_body():
                nonlocal consumed
                if not consumed:
                    consumed = True
                    return {'type': 'http.request', 'body': body, 'more_body': False}
                return await receive()
            return await self.app(scope, replay_body, secure_send)
        await self.app(scope, receive, secure_send)
'''
# END GENERATED API SECURITY
import asyncio, math, sys
from collections import deque
from datetime import datetime, timezone
from typing import Annotated, Literal
import chess
from fastapi import FastAPI, HTTPException, Request
from fastapi.exceptions import RequestValidationError
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel, ConfigDict, Field, StringConstraints, field_validator
from concurrent.futures import ThreadPoolExecutor
from starlette.exceptions import HTTPException as StarletteHTTPException

# Stop the old app before replacing route globals on a rerun.
if callable(globals().get('stop_server')):
    stop_server()
API_STARTED = time.monotonic()
API_LOG = deque(maxlen=50)
API_COUNTERS = {'requests': 0, 'success': 0, 'errors': 0, 'fallbacks': 0}
API_STATS_LOCK = threading.Lock()
if globals().get('API_EXECUTOR') is not None:
    API_EXECUTOR.shutdown(wait=False, cancel_futures=True)
API_EXECUTOR = ThreadPoolExecutor(max_workers=2, thread_name_prefix='chess-search')

import types, secrets
API_SECURITY = types.ModuleType('chess_api_security')
exec(compile(SECURITY_SOURCE, '<api-security>', 'exec'), API_SECURITY.__dict__)
API_POLICY = API_SECURITY.PublicPolicy(
    API_SECURITY.allowed_origins(WEBSITE_ORIGINS), trust_cloudflare=TUNNEL == 'cloudflare')

class MoveRequest(BaseModel):
    model_config = ConfigDict(extra='forbid', strict=True)
    fen: str = Field(max_length=200)
    history: list[Annotated[str, StringConstraints(min_length=4, max_length=5)]] = Field(default_factory=list, max_length=1024)
    difficulty: Literal['easy', 'medium', 'hard'] = 'medium'

    @field_validator('fen')
    @classmethod
    def valid_fen(cls, value):
        try:
            board = chess.Board(value)
            if len(value.split()) != 6 or not board.is_valid():
                raise ValueError('invalid board')
        except ValueError:
            raise HTTPException(400, 'Invalid FEN: provide a valid six-field chess position.')
        return value

    @field_validator('history')
    @classmethod
    def valid_history(cls, value):
        for uci in value:
            try:
                move = chess.Move.from_uci(uci)
                if not move or move.drop:
                    raise ValueError('not a standard move')
            except ValueError:
                raise ValueError('History must contain standard UCI moves, such as e2e4.')
        return value

api = FastAPI(title='Chess API', version=API_VERSION, docs_url=None, redoc_url=None, openapi_url=None)

@api.exception_handler(StarletteHTTPException)
async def http_error(_request, exc):
    return JSONResponse(status_code=exc.status_code, content={'error': str(exc.detail)}, headers=exc.headers)

@api.exception_handler(RequestValidationError)
async def validation_error(_request, exc):
    message = 'Invalid request: check FEN, UCI history (up to 1024 moves), and difficulty.'
    return JSONResponse(status_code=422, content={'error': message})

@api.middleware('http')
async def record_request(request: Request, call_next):
    if request.method != 'POST' or request.url.path != '/move':
        return await call_next(request)
    started = time.monotonic()
    entry = {'fen': None, 'difficulty': None, 'move': None}
    try:
        body = await request.json()
        if isinstance(body, dict):
            entry.update(fen=body.get('fen')[:200] if isinstance(body.get('fen'), str) else None,
                         difficulty=body.get('difficulty', 'medium')[:16] if isinstance(body.get('difficulty', 'medium'), str) else None)
    except (ValueError, UnicodeDecodeError):
        pass
    try:
        response = await call_next(request)
    except Exception:
        response = JSONResponse(status_code=500, content={'error': 'Engine failed. Retry or check the notebook.'})
    entry.update(getattr(request.state, 'move_log', {}))
    entry.update(status=response.status_code, time_ms=int((time.monotonic() - started) * 1000),
                 at=datetime.now(timezone.utc).isoformat())
    with API_STATS_LOCK:
        API_LOG.append(entry)
        API_COUNTERS['requests'] += 1
        API_COUNTERS['success' if response.status_code < 400 else 'errors'] += 1
        API_COUNTERS['fallbacks'] += int(entry.get('fallback', False))
    return response

# Outermost middleware also adds CORS headers to errors, including custom headers.
api.add_middleware(CORSMiddleware, allow_origins=list(API_POLICY.origins),
                   allow_methods=['GET', 'POST'], allow_headers=['Content-Type', 'Authorization', 'ngrok-skip-browser-warning'],
                   expose_headers=['Retry-After'], max_age=600)
api.add_middleware(API_SECURITY.SecurityBoundary, policy=API_POLICY)

@api.get('/health')
async def health():
    return {'status': 'ok', 'engine': 'chess', 'version': API_VERSION,
            'uptime_s': int(time.monotonic() - API_STARTED)}

@api.get('/stats')
async def stats(request: Request):
    expected = 'Bearer ' + API_ADMIN_TOKEN
    supplied = request.headers.get('authorization', '')
    if not secrets.compare_digest(supplied.encode('utf-8'), expected.encode('utf-8')):
        raise HTTPException(401, 'Administrator authorization required.', headers={'WWW-Authenticate': 'Bearer'})
    with API_STATS_LOCK:
        return {'counters': dict(API_COUNTERS), 'last_requests': list(API_LOG), 'security': API_POLICY.snapshot()}

class ApiSearchDeadline(Exception):
    pass

def depth_one_fallback(board):
    """Score every legal successor with the exported evaluation; no random fallback."""
    moves = list(board.legal_moves)
    children = []
    for move in moves:
        child = board.copy(stack=True)
        child.push(move)
        children.append(child)
    if hasattr(API_ENGINE, 'values'):  # GPU export: batched neural evaluation.
        scores = [-float(value) for value in API_ENGINE.values(children)]
    else:
        genome = API_ENGINE.nn if API_ENGINE.nn is not None else API_ENGINE.genome
        scores = [-float(API_MODULE.evaluate(child, genome)) for child in children]
    for i, child in enumerate(children):
        if child.is_checkmate():
            scores[i] = 30000.0
        elif child.is_game_over():
            scores[i] = 0.0
    best = max(range(len(moves)), key=lambda i: scores[i])
    return {'move': moves[best].uci(), 'info': {'eval': scores[best], 'depth': 1,
            'nodes': len(moves), 'fallback': True}}

def search_locked(payload, board, deadline):
    # Waiting for the lock counts against the request budget too.
    remaining = deadline - time.monotonic()
    if remaining <= 0 or not API_ENGINE_LOCK.acquire(timeout=max(0, remaining)):
        raise ApiSearchDeadline()
    old_trace = sys.gettrace()
    limits = API_ENGINE.settings[payload.difficulty]
    old_seconds = limits['seconds']
    def check_deadline(frame, event, arg):
        if frame.f_code is not search_locked.__code__ and time.monotonic() >= deadline:
            raise ApiSearchDeadline()
        return check_deadline
    try:
        limits['seconds'] = min(float(old_seconds), max(.001, deadline - time.monotonic()))
        # Existing search deadlines plus a Python deadline cover preprocessing/fallback.
        # Native NumPy/Torch calls finish before Python can observe this deadline.
        sys.settrace(check_deadline)
        try:
            result = API_ENGINE.get_move(payload.fen, payload.history, payload.difficulty)
        except AssertionError:
            result = {}  # Export assertions may catch an illegal move before returning it.
        try:
            move = chess.Move.from_uci(result.get('move', ''))
        except (ValueError, TypeError, AttributeError):
            move = None
        if move not in board.legal_moves:
            result = depth_one_fallback(board)
        info = result.get('info', {})
        score = float(info.get('eval', 0))
        if not math.isfinite(score):
            raise ValueError('Engine returned a non-finite score')
        # Exports score in side-to-move centipawns. HTTP scores are White pawns.
        info = {**info, 'eval': score * (1 if board.turn == chess.WHITE else -1) / 100,
                'depth': int(info.get('depth', 0)), 'nodes': int(info.get('nodes', 0))}
        return {'move': result['move'], 'info': info}
    finally:
        sys.settrace(old_trace)
        limits['seconds'] = old_seconds
        API_ENGINE_LOCK.release()

@api.post('/move')
async def move(payload: MoveRequest, request: Request):
    board = chess.Board(payload.fen)
    if payload.history:
        replay = chess.Board()
        try:
            for uci in payload.history:
                replay.push_uci(uci)
        except ValueError:
            raise HTTPException(400, 'Invalid history: moves must be legal from the starting position.')
        if replay.fen() != board.fen():
            raise HTTPException(400, 'History does not match the supplied FEN.')
        board = replay
    if board.is_game_over() or board.is_repetition(3) or board.is_fifty_moves():
        raise HTTPException(400, 'Game over: this position has no move to play.')
    started = time.monotonic()
    cap = API_TIME_CAPS[payload.difficulty]
    slots = API_POLICY.search_slots
    if not slots.acquire(blocking=False):
        with API_POLICY.lock:
            API_POLICY.counts['busy'] += 1
        raise HTTPException(503, 'The AI is busy. Retry shortly.', headers={'Retry-After': '2'})
    try:
        future = API_EXECUTOR.submit(search_locked, payload, board, started + cap)
    except BaseException:
        slots.release()
        raise
    # Release only when the actual worker finishes (or never-started work is cancelled).
    # HTTP cancellation must never release admission while native code is still running.
    future.add_done_callback(lambda _future: slots.release())
    task = asyncio.wrap_future(future)
    task.add_done_callback(lambda t: t.exception() if not t.cancelled() else None)
    # Keep the lock in the worker even if HTTP times out: never overlap engine calls.
    done, _ = await asyncio.wait({task}, timeout=cap)
    if not done:
        # Consume a late worker exception. The trace stops Python work on its next line.
        task.add_done_callback(lambda t: t.exception() if not t.cancelled() else None)
        raise HTTPException(504, f'Chess exceeded the {cap:g}s {payload.difficulty} time cap. Retry.')
    try:
        result = task.result()
    except ApiSearchDeadline:
        raise HTTPException(504, f'Chess exceeded the {cap:g}s {payload.difficulty} time cap. Retry.')
    except Exception:
        raise HTTPException(500, 'Engine failed. Retry or check the exported model files.')
    # Verify again at the API boundary before returning anything to the browser.
    if chess.Move.from_uci(result['move']) not in board.legal_moves:
        raise HTTPException(500, 'Engine could not produce a legal move.')
    result['info']['time_ms'] = int((time.monotonic() - started) * 1000)
    request.state.move_log = {'move': result['move'], 'fallback': result['info'].get('fallback', False)}
    return result

print('API ready: public /health and /move; /stats requires the runtime-only admin token.')


API ready: public /health and /move; /stats requires the runtime-only admin token.


In [5]:
#@title 3 · Start the API and run its self-tests { display-mode: "form" }
import requests, uvicorn, nest_asyncio
nest_asyncio.apply()

def stop_server():
    server = globals().get('API_SERVER')
    thread = globals().get('API_SERVER_THREAD')
    if server is not None:
        server.should_exit = True
    if thread is not None and thread.is_alive():
        thread.join(timeout=5)
        if thread.is_alive():
            server.force_exit = True
            thread.join(timeout=2)
        if thread.is_alive():
            raise RuntimeError('Previous server is still stopping. Wait and rerun this cell.')
    # A timed-out native call can still own the engine; do not replace its lock.
    if not API_ENGINE_LOCK.acquire(timeout=2):
        raise RuntimeError('Engine is still finishing a request. Wait and rerun this cell.')
    API_ENGINE_LOCK.release()

stop_server()
API_SERVER = uvicorn.Server(uvicorn.Config(api, host='127.0.0.1', port=8000,
                           log_level='warning', loop='asyncio', timeout_graceful_shutdown=3,
                           proxy_headers=False, server_header=False, limit_concurrency=32,
                           timeout_keep_alive=5, h11_max_incomplete_event_size=16384))
API_SERVER_THREAD = threading.Thread(target=API_SERVER.run, daemon=True, name='cristopher-api')
API_SERVER_THREAD.start()
for _ in range(100):
    if API_SERVER.started:
        break
    if not API_SERVER_THREAD.is_alive():
        raise RuntimeError('Server failed to start. Port 8000 may already be in use.')
    time.sleep(.1)
else:
    stop_server()
    raise RuntimeError('Server did not become ready on port 8000.')

def api_self_test(base_url, public=False):
    headers = {'ngrok-skip-browser-warning': 'true', 'Origin': 'https://deployment-check.example' if '*' in API_POLICY.origins else API_POLICY.origins[0]}
    health_response = requests.get(base_url + '/health', headers=headers, timeout=4)
    health_response.raise_for_status()
    assert health_response.json()['status'] == 'ok'
    response = requests.post(base_url + '/move', headers=headers, timeout=15,
        json={'fen': chess.STARTING_FEN, 'history': [], 'difficulty': 'easy'})
    response.raise_for_status()
    assert chess.Move.from_uci(response.json()['move']) in chess.Board().legal_moves
    print('OK: health + legal starting-position move' + (' through PUBLIC URL' if public else ' locally'))
    if not public:
        for fen, label in [('not a FEN', 'invalid FEN'), ('7k/5Q2/6K1/8/8/8/8/8 b - - 0 1', 'finished game')]:
            response = requests.post(base_url + '/move', headers=headers, timeout=4,
                                     json={'fen': fen, 'difficulty': 'medium'})
            assert response.status_code == 400, response.text
            assert response.json().get('error'), response.text
            print(f'OK: {label} -> 400: {response.json()["error"]}')

api_self_test('http://127.0.0.1:8000')
print('Server running in background on port 8000. Use stop_server() to stop it.')


def shutdown_api():
    try:
        if callable(globals().get('stop_tunnel')):
            stop_tunnel()
    finally:
        stop_server()
    API_EXECUTOR.shutdown(wait=False, cancel_futures=True)
    print('API and tunnel stopped.')


OK: health + legal starting-position move locally
OK: invalid FEN -> 400: Invalid FEN: provide a valid six-field chess position.
OK: finished game -> 400: Game over: this position has no move to play.
Server running in background on port 8000. Use stop_server() to stop it.


## Use the URL
The last cell prints the verified HTTPS API URL. In the game, open **Settings → Server URL**, paste it and enable **Use AI server**. Select AI and press Play. Restarting deployment can change the URL. Public connection tests retry every three seconds until they succeed; interrupt the Colab cell to stop retries and close the tunnel.

WEBSITE_ORIGINS defaults to **`*`**, allowing any website origin. You can optionally restrict it to exact comma-separated origins without paths. Public players need no shared password. Rate limits, bounded request bodies, two admitted searches, time caps and the configured browser-origin policy apply. Busy/rate-limited clients receive 503/429 with Retry-After.

`/stats` is private. Inspect `API_COUNTERS`, `API_LOG`, or `API_POLICY.snapshot()` inside this notebook. Do not publish API_ADMIN_TOKEN or notebook outputs. To stop, call `shutdown_api()` or disconnect the runtime.

GitHub Pages hosts only static files; Colab runs the API. Application limits reduce abuse but do not guarantee uptime against distributed attacks. Colab and quick tunnels remain temporary, limited-capacity hosting. See `docs/PUBLIC_DEPLOYMENT.md` for the publishing workflow and limitations.


In [8]:
#@title 4 · Open the tunnel and print the verified URL { display-mode: "form" }
import re, queue, urllib.request, html
from itertools import count
from IPython.display import display, HTML

def stop_tunnel():
    global PUBLIC_URL
    process = globals().get('CLOUDFLARED_PROCESS')
    if process is not None:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=3)
        reader = globals().get('CLOUDFLARED_READER')
        if reader is not None:
            reader.join(timeout=2)
        if process.stdout:
            process.stdout.close()
    tunnel = globals().get('NGROK_TUNNEL')
    if tunnel is not None:
        from pyngrok import ngrok
        ngrok.disconnect(tunnel.public_url)
        ngrok.kill()
        globals()['NGROK_TUNNEL'] = None
    PUBLIC_URL = None


def wait_for_public_api(base_url):
    # Retry until healthy, but let Colab interruption propagate to tunnel cleanup.
    for attempt in count(1):
        try:
            api_self_test(base_url, public=True)
            return
        except (requests.RequestException, ValueError, AssertionError) as error:
            print(f'Public test {attempt}: FAIL ({type(error).__name__}); retrying in 3 seconds…')
            time.sleep(3)

stop_tunnel()
try:
    if TUNNEL == 'cloudflare':
        binary = Path('/content/cloudflared')
        # Pinned official release; verify even an existing cached executable.
        import hashlib
        CLOUDFLARED_URL = 'https://github.com/cloudflare/cloudflared/releases/download/2026.10.5/cloudflared-linux-amd64'
        CLOUDFLARED_SHA256 = 'e1bc0cf084fd9e590554fe769a86b1db46701c3f847a43d902f7b974a4e16298'
        if not binary.exists() or hashlib.sha256(binary.read_bytes()).hexdigest() != CLOUDFLARED_SHA256:
            download = binary.with_suffix('.download')
            urllib.request.urlretrieve(CLOUDFLARED_URL, download)
            if hashlib.sha256(download.read_bytes()).hexdigest() != CLOUDFLARED_SHA256:
                download.unlink(missing_ok=True)
                raise RuntimeError('Tunnel binary checksum mismatch; refusing to execute it.')
            download.chmod(0o755)
            download.replace(binary)
        tunnel_lines = queue.Queue(maxsize=1)
        tunnel_tail = deque(maxlen=30)
        CLOUDFLARED_PROCESS = subprocess.Popen(
            [str(binary), 'tunnel', '--url', 'http://localhost:8000'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        def read_tunnel_output(process, output, tail):
            # Drain continuously so cloudflared cannot block on a full output pipe.
            for line in process.stdout:
                tail.append(line.rstrip())
                match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com\b', line)
                if match:
                    try:
                        output.put_nowait(match.group(0))
                    except queue.Full:
                        pass
        CLOUDFLARED_READER = threading.Thread(target=read_tunnel_output,
            args=(CLOUDFLARED_PROCESS, tunnel_lines, tunnel_tail), daemon=True)
        CLOUDFLARED_READER.start()
        try:
            PUBLIC_URL = tunnel_lines.get(timeout=45)
        except queue.Empty:
            raise RuntimeError('Cloudflare did not print a URL. Check network/config:\n' + '\n'.join(tunnel_tail))
    elif TUNNEL == 'ngrok':
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'])
        from pyngrok import ngrok, conf
        from getpass import getpass
        try:
            from google.colab import userdata
            token = userdata.get('NGROK_TOKEN')
        except Exception:
            token = None
        if not token:
            token = getpass('Ngrok auth token (hidden): ')
        # Pass in-memory configuration; never print or hardcode the secret.
        ngrok_config = conf.PyngrokConfig(auth_token=token)
        try:
            NGROK_TUNNEL = ngrok.connect(8000, 'http', bind_tls=True, pyngrok_config=ngrok_config)
        except Exception:
            # Do not echo provider exceptions that could contain credentials.
            raise RuntimeError('Ngrok failed. Check NGROK_TOKEN and your ngrok account.') from None
        finally:
            token = None
            ngrok_config.auth_token = None
        PUBLIC_URL = NGROK_TUNNEL.public_url
    else:
        raise ValueError('TUNNEL must be cloudflare or ngrok')
    wait_for_public_api(PUBLIC_URL)
    display(HTML(f'<h2>API ready</h2><p><strong>{html.escape(PUBLIC_URL)}</strong></p>'))
    print(f'Paste this URL in the web page → Server URL: {PUBLIC_URL}')
except BaseException:
    stop_tunnel()
    raise


Public test 1/11: FAIL (ConnectionError); retrying…
Public test 2/11: FAIL (ConnectionError); retrying…
Public test 3/11: FAIL (ConnectionError); retrying…
Public test 4/11: FAIL (ConnectionError); retrying…
Public test 5/11: FAIL (ConnectionError); retrying…
Public test 6/11: FAIL (ConnectionError); retrying…
Public test 7/11: FAIL (ConnectionError); retrying…
Public test 8/11: FAIL (ConnectionError); retrying…
Public test 9/11: FAIL (ConnectionError); retrying…
Public test 10/11: FAIL (ConnectionError); retrying…
OK: health + legal starting-position move through PUBLIC URL


Paste this URL in the web page → Server URL: https://interpretation-specifications-mat-transmitted.trycloudflare.com
